# OpenClaw Provider Routing (OpenAI + OLLAMA)

This notebook shows two simple OpenClaw provider use cases:
1. Configure OpenClaw to use OpenAI as the primary model provider
2. Configure OpenClaw to use OLLAMA via `host.docker.internal` when running inside Docker

References:
- https://docs.openclaw.ai/providers/openai
- https://docs.openclaw.ai/providers/ollama


## Docker note

When notebooks run in Docker on macOS, use host networking alias for OLLAMA:

```yaml
services:
  notebooks:
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - OLLAMA_BASE_URL=http://host.docker.internal:11434
```

Why this works: inside the container, `localhost` points to the container itself, not your Mac host.
`host.docker.internal` is the Docker Desktop host alias that routes requests from container to your machine, where Ollama is running on port `11434`.

You do not need to install the `ollama` CLI in the notebook container for HTTP-based model calls.


In [ ]:
import os
import shlex
import shutil
import subprocess

HAS_OPENCLAW = shutil.which("openclaw") is not None
IN_DOCKER = os.path.exists('/.dockerenv')
print(f"openclaw available: {HAS_OPENCLAW}")
print(f"running in docker: {IN_DOCKER}")
print(f"OPENAI_API_KEY set: {bool(os.getenv('OPENAI_API_KEY'))}")
print(f"OLLAMA_BASE_URL: {os.getenv('OLLAMA_BASE_URL', 'http://host.docker.internal:11434')}")


In [ ]:
def run_cmd(args, *, env=None):
    cmd = [str(a) for a in args]
    print("$", " ".join(shlex.quote(x) for x in cmd))
    proc = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if proc.stdout.strip():
        print(proc.stdout.strip())
    if proc.stderr.strip():
        print(proc.stderr.strip())
    return proc.returncode


## Use case 1: OpenClaw with OpenAI

This configures OpenClaw to use an OpenAI model reference (`openai/...`) as default.


In [ ]:
if not HAS_OPENCLAW:
    print("Skipping: openclaw CLI is not installed in this environment.")
elif not os.getenv('OPENAI_API_KEY'):
    print("Skipping: OPENAI_API_KEY is not set.")
else:
    if IN_DOCKER:
        print("Skipping 'openclaw gateway status' in Docker (systemd is unavailable in containers).")
    else:
        run_cmd(["openclaw", "gateway", "status"])
    run_cmd(["openclaw", "config", "set", "agents.defaults.model.primary", "openai/gpt-5.1-codex"])
    run_cmd(["openclaw", "models", "list"])
    print("OpenAI provider configuration complete.")


## Use case 2: OpenClaw with OLLAMA (small local model)

This configures OpenClaw to use OLLAMA as primary and keeps model size small.


In [ ]:
if not HAS_OPENCLAW:
    print("Skipping: openclaw CLI is not installed in this environment.")
else:
    ollama_base = os.getenv('OLLAMA_BASE_URL', 'http://host.docker.internal:11434')
    run_cmd(["openclaw", "config", "set", "models.providers.ollama.baseUrl", ollama_base])
    run_cmd(["openclaw", "config", "set", "models.providers.ollama.apiKey", "ollama-local"])
    run_cmd(["openclaw", "config", "set", "agents.defaults.model.primary", "ollama/qwen2.5-coder:1.5b"])
    run_cmd(["openclaw", "models", "list"])
    print("OLLAMA provider configuration complete.")


## Next steps

- Open dashboard: `openclaw dashboard`
- Re-run model selection as needed with `openclaw configure --section models`
- For gateway health checks on a host OS: `openclaw gateway status`
- In Docker notebooks, skip service checks that depend on `systemd`
